# EEGText D1 — complete ZuCo download and audit

This notebook downloads the 24 official ZuCo 1.0 NR and TSR subject files selected by D0, validates their exact byte sizes, audits both tasks, and rebuilds the combined SR+NR+TSR manifest. The planned Drive use is 17.44 GiB: 9.64 GiB for NR and 7.80 GiB for TSR.

Each completed file is permanent in Drive. If the runtime disconnects, rerun Cells 1 and 2, then rerun the interrupted download cell. A `.part` file is resumed when the server supports byte ranges.

In [ ]:
# 1) Fetch the project and run download-free tests.
from pathlib import Path
import json, os, shutil, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)

EEGTEXT_ROOT = PROJECT_ROOT / "eegtext"
os.chdir(EEGTEXT_ROOT)
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"])


In [ ]:
# 2) Mount Drive, validate D0, and show the exact remaining download.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
DATA_ROOT = THESIS_ROOT / "Data"
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/eegtext"
MANIFEST_ROOT = CACHE_ROOT / "corpus_manifests"
DOWNLOAD_STATUS_ROOT = CACHE_ROOT / "downloads"
INVENTORY_JSON = MANIFEST_ROOT / "official_osf_zuco_1/inventory.json"
TASK_DIRS = {
    "SR": DATA_ROOT / "zuco_og_raw",
    "NR": DATA_ROOT / "zuco_1_task2_nr",
    "TSR": DATA_ROOT / "zuco_1_task3_tsr",
}
if not INVENTORY_JSON.exists():
    raise FileNotFoundError(f"Run D0 first; inventory is missing: {INVENTORY_JSON}")
if not (MANIFEST_ROOT / "zuco_1_sr/recordings.csv").exists():
    raise FileNotFoundError("Run the D0 Task 1 audit before this notebook.")

from src.download import task_plan
plans = {task: task_plan(INVENTORY_JSON, task, TASK_DIRS[task]) for task in ("NR", "TSR")}
remaining_bytes = 0
for task, plan in plans.items():
    remaining = 0
    for item in plan["files"]:
        destination = Path(item["destination"])
        partial = destination.with_suffix(destination.suffix + ".part")
        if destination.exists():
            if destination.stat().st_size != item["size_bytes"]:
                raise ValueError(f"Wrong-size existing file: {destination}")
            continue
        partial_bytes = partial.stat().st_size if partial.exists() else 0
        remaining += item["size_bytes"] - partial_bytes
    remaining_bytes += remaining
    print(f"{task}: {plan['file_count']} files, {plan['total_bytes'] / 1024**3:.2f} GiB total, {remaining / 1024**3:.2f} GiB remaining")

free_bytes = shutil.disk_usage(DATA_ROOT).free
print(f"Drive reports {free_bytes / 1024**3:.2f} GiB free; {remaining_bytes / 1024**3:.2f} GiB remains to download.")
if free_bytes < remaining_bytes:
    raise RuntimeError("Drive does not report enough free space for the remaining files.")


In [ ]:
# 3) Download and validate all 12 NR files (9.64 GiB total).
run([
    sys.executable, "run.py", "download-zuco-task",
    "--inventory", str(INVENTORY_JSON),
    "--task", "NR",
    "--output-dir", str(TASK_DIRS["NR"]),
    "--status-file", str(DOWNLOAD_STATUS_ROOT / "zuco_1_nr.json"),
])


In [ ]:
# 4) Audit NR immediately after its download completes.
run([
    sys.executable, "run.py", "audit-zuco",
    "--raw-dir", str(TASK_DIRS["NR"]),
    "--dataset", "zuco",
    "--release", "1.0",
    "--task", "NR",
    "--pattern", "results*_NR.mat",
    "--output-dir", str(MANIFEST_ROOT / "zuco_1_nr"),
])
print(json.dumps(json.loads((MANIFEST_ROOT / "zuco_1_nr/summary.json").read_text()), indent=2))


In [ ]:
# 5) Download and validate all 12 TSR files (7.80 GiB total).
run([
    sys.executable, "run.py", "download-zuco-task",
    "--inventory", str(INVENTORY_JSON),
    "--task", "TSR",
    "--output-dir", str(TASK_DIRS["TSR"]),
    "--status-file", str(DOWNLOAD_STATUS_ROOT / "zuco_1_tsr.json"),
])


In [ ]:
# 6) Audit TSR immediately after its download completes.
run([
    sys.executable, "run.py", "audit-zuco",
    "--raw-dir", str(TASK_DIRS["TSR"]),
    "--dataset", "zuco",
    "--release", "1.0",
    "--task", "TSR",
    "--pattern", "results*_TSR.mat",
    "--output-dir", str(MANIFEST_ROOT / "zuco_1_tsr"),
])
print(json.dumps(json.loads((MANIFEST_ROOT / "zuco_1_tsr/summary.json").read_text()), indent=2))


In [ ]:
# 7) Combine SR, NR, and TSR and report cross-task duplicate text.
manifest_paths = [MANIFEST_ROOT / f"zuco_1_{task.lower()}/recordings.csv" for task in ("SR", "NR", "TSR")]
for path in manifest_paths:
    if not path.exists():
        raise FileNotFoundError(f"Task audit is incomplete: {path}")
command = [sys.executable, "run.py", "combine-manifests"]
for path in manifest_paths:
    command.extend(["--manifest", str(path)])
command.extend(["--output-dir", str(MANIFEST_ROOT / "zuco_1_combined")])
run(command)

print("Final task summaries:")
for task in ("sr", "nr", "tsr", "combined"):
    summary = json.loads((MANIFEST_ROOT / f"zuco_1_{task}/summary.json").read_text())
    print(f"\n{task.upper()}")
    print(json.dumps(summary, indent=2))
print("\nEverything is saved in Drive. The runtime can now be disconnected safely.")


## After Cell 7

You do not need to copy the long output. The raw files, download reports, task manifests, and combined summary are all persistent in Drive and can be inspected directly.